In [0]:
from pyspark.sql.functions import count, max, min, avg, sum, round
from modules.utils.date_utils import get_month_start_n_months_ago

In [0]:
# Get the first day of the month six months ago
six_months_ago_start = get_month_start_n_months_ago(6)

In [0]:
# Load the enriched trip dataset and filter to only include trips with a pickup datetime later than the start the date from six months ago months ago
df = spark.read.table("nyctaxi.02_silver.yellow_trips_enriched").filter(f"tpep_pickup_datetime > '{six_months_ago_start}'")

In [0]:
# Aggregate trip data by pickup date with metrics
# Group records by calendar date
df = df.groupBy(df.tpep_pickup_datetime.cast("date").alias("pickup_date")).\
    agg(count("*").alias("total_trips"), 
        round(avg("passenger_count"), 1).alias("average_passengers"),
        round(avg("trip_distance"), 1).alias("average_distance"),
        round(avg("fare_amount"), 2).alias("average_fare_per_trip"),
        max("fare_amount").alias("max_fare"),
        min("fare_amount").alias("min_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )


In [0]:
df.write.mode("append").saveAsTable("nyctaxi.03_gold.daily_trip_summary")